# Hanoi (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Hanoi_java.ipynb)

Chapter 14 — Recursion. Runnable notebook of `Hanoi`: the algorithm in Java, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Hanoi_python.ipynb) and [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Hanoi_cpp.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion_java.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. The first cell checks that the JDK is there and installs it if it is missing. Each Java program is written to a file by a `%%writefile` cell, then compiled with `javac` and run with `java` in the next cell.

**Names.** There are no arrays in this chapter: variables and parameters have the same names as in the pseudo-code. The stack and the counters are `static` fields of the class, and the trace is switched on with `trace = true`.

## Setup

On Colab the JDK is usually already there: the cell below checks it and installs it if it is missing.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Towers of Hanoi: Hanoi

**Input:** the number of disks $n \ge 1$, the source peg $from$, the destination peg $to$ and the auxiliary peg $aux$. **Output:** the sequence of moves that transfers disks $1, \dots, n$ from $from$ to $to$. $M(n) = 2^n - 1$ moves (optimal), $S(n) = 3 \cdot 2^n - 4 = \Theta(2^n)$ steps, depth $n$.

Pseudo-code: section 14.8 of the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

In [2]:
%%writefile Hanoi.java
public class Hanoi
{
static long[] cnt = new long[20];   // cnt[r] = number of executions of line r (numbering of the lecture notes)
static long calls = 0;              // number of calls = nodes of the tree of recursive calls
static int maxDepth = 0;            // depth of the recursion = maximum height of the stack
static long mults = 0;              // multiplications (FastPower)
static long moves = 0;              // moves (Hanoi)
static int ev = 0;                  // counter of the events of the trace
static java.util.ArrayList<Integer> stack = new java.util.ArrayList<>();   // the call stack: the arguments of the active calls
static long[] seen = new long[64];  // seen[x] = number of calls with argument x
static boolean trace = false;       // true: print the trace of the execution

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
    java.util.Arrays.fill(seen, 0);
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0;
    stack.clear();
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Line | Executions");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Total steps S = " + total());
}

static void push(int x)             // a call starts: its frame goes on top of the stack
{
    calls++;
    stack.add(x);
    if (stack.size() > maxDepth) maxDepth = stack.size();
    if (x >= 0 && x < 64) seen[x]++;
}

static void pop()                   // the call returns: its frame leaves the stack
{
    stack.remove(stack.size() - 1);
}

static void event(String text)      // numbered line of the trace, with the stack
{
    ev++;
    System.out.println(String.format("%2d. %s   stack: %s", ev, text, stack));
}

static void say(String call, String action)   // line of the trace: the call, indented one level for every active call
{
    System.out.println(String.format("%-28s| %s", "  ".repeat(stack.size() - 1) + call, action));
}

    static boolean showMoves = false;              // true: print the moves

    static void move(int disk, String from, String to)   // "move disk d from from to to": counts the move and, if showMoves, prints it
    {
        moves++;
        if (showMoves) System.out.println(String.format("%d. move disk %d from %s to %s", moves, disk, from, to));
    }

    // Hanoi(n, from, to, aux)
    static void hanoi(int n, String from, String to, String aux)
    {
        push(n); cnt[2]++;
        String call = trace ? String.format("Hanoi(%d,%s,%s,%s)", n, from, to, aux) : "";
        if (n == 1)
        {
            cnt[3]++;
            move(1, from, to);
            if (trace) say(call, String.format("prints: move disk 1 from %s to %s, returns", from, to));
        }
        else
        {
            if (trace) say(call, String.format("calls Hanoi(%d,%s,%s,%s)", n - 1, from, aux, to));
            cnt[5]++;
            hanoi(n - 1, from, aux, to);
            cnt[6]++;
            move(n, from, to);
            if (trace) say(call, String.format("prints: move disk %d from %s to %s, calls Hanoi(%d,%s,%s,%s)", n, from, to, n - 1, aux, to, from));
            cnt[7]++;
            hanoi(n - 1, aux, to, from);
            if (trace) say(call, "returns");
        }
        pop();
    }

    public static void main(String[] args)
    {
        // Example of the lecture notes: Hanoi(3, A, C, B), the moves
        reset();
        showMoves = true;
        hanoi(3, "A", "C", "B");
        showMoves = false;
        System.out.printf("%d moves, %d calls, depth of the recursion = %d\n", moves, calls, maxDepth);

        // The same execution, call by call (indentation = depth in the stack)
        System.out.printf("\n");
        reset();
        trace = true;
        System.out.println(String.format("%-28s| %s", "call", "action"));
        hanoi(3, "A", "C", "B");
        trace = false;
        printSteps(2, 3, 5, 6, 7);
        System.out.printf("S(n) = 3 * 2^n - 4 = %d\n", 3 * (1 << 3) - 4);

        // Moves and steps: M(n) = 2^n - 1 and S(n) = 3 * 2^n - 4
        System.out.printf("\n");
        System.out.printf("  n |  moves | 2^n - 1 |    calls |   S(n) | 3 * 2^n - 4 | depth\n");
        for (int k = 1; k <= 10; k++)
        {
            reset();
            hanoi(k, "A", "C", "B");
            System.out.printf("%3d | %6d | %7d | %8d | %6d | %11d | %d\n", k, moves, (1 << k) - 1, calls, total(), 3 * (1 << k) - 4, maxDepth);
        }

        // The legend: 64 disks, one move per second
        System.out.printf("\n");
        System.out.printf("n = 64: 2^64 - 1 seconds are more than %.1e years\n", (Math.pow(2.0, 64) - 1) / (365.25 * 24 * 3600));
    }
}

Writing Hanoi.java


In [3]:
!javac Hanoi.java && java Hanoi

1. move disk 1 from A to C
2. move disk 2 from A to B
3. move disk 1 from C to B
4. move disk 3 from A to C
5. move disk 1 from B to A
6. move disk 2 from B to C
7. move disk 1 from A to C
7 moves, 7 calls, depth of the recursion = 3

call                        | action
Hanoi(3,A,C,B)              | calls Hanoi(2,A,B,C)
  Hanoi(2,A,B,C)            | calls Hanoi(1,A,C,B)
    Hanoi(1,A,C,B)          | prints: move disk 1 from A to C, returns
  Hanoi(2,A,B,C)            | prints: move disk 2 from A to B, calls Hanoi(1,C,B,A)
    Hanoi(1,C,B,A)          | prints: move disk 1 from C to B, returns
  Hanoi(2,A,B,C)            | returns
Hanoi(3,A,C,B)              | prints: move disk 3 from A to C, calls Hanoi(2,B,C,A)
  Hanoi(2,B,C,A)            | calls Hanoi(1,B,A,C)
    Hanoi(1,B,A,C)          | prints: move disk 1 from B to A, returns
  Hanoi(2,B,C,A)            | prints: move disk 2 from B to C, calls Hanoi(1,A,C,B)
    Hanoi(1,A,C,B)          | prints: move disk 1 from A to C, returns
 

**Try it yourself.** Change the number of disks and count the moves.

In [4]:
%%writefile ProvaHanoi.java
public class ProvaHanoi
{
    public static void main(String[] args)
    {
        int n = 4;        // <-- change the number of disks
        Hanoi.reset();
        Hanoi.showMoves = true;
        Hanoi.hanoi(n, "A", "C", "B");
        Hanoi.showMoves = false;
        System.out.printf("%d moves = 2^%d - 1 = %d\n", Hanoi.moves, n, (1 << n) - 1);
    }
}

Writing ProvaHanoi.java


In [5]:
!javac ProvaHanoi.java && java ProvaHanoi

1. move disk 1 from A to B
2. move disk 2 from A to C
3. move disk 1 from B to C
4. move disk 3 from A to B
5. move disk 1 from C to A
6. move disk 2 from C to B
7. move disk 1 from A to B
8. move disk 4 from A to C
9. move disk 1 from B to C
10. move disk 2 from B to A
11. move disk 1 from C to A
12. move disk 3 from B to C
13. move disk 1 from A to B
14. move disk 2 from A to C
15. move disk 1 from B to C
15 moves = 2^4 - 1 = 15


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).